In [40]:
import pandas as pd
import numpy as np
import glob, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, classification_report, confusion_matrix

In [33]:
globbed = glob.glob(os.path.join('TrafficLabelling', '*.csv'))
# encoding = 'latin1' from Gemini, otherwise encountering error with decoding CSV data
dfs = [pd.read_csv(file, encoding='latin1', low_memory=False) for file in globbed]
data = pd.concat(dfs, ignore_index=True)

In [ ]:
cleaned_columns = []
for col in data.columns:
    cleaned_columns.append(col.strip().lower().replace(' ', '_'))
data.columns = cleaned_columns

data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
data['malicious'] = (data['label'] != 'BENIGN').astype(int)
data.drop(columns=['label'], inplace=True)
data.replace([np.inf, -np.inf], np.nan, inplace=True)

data.dropna(inplace=True)
data.sort_values('timestamp', inplace=True)


In [ ]:
index = int(len(data) * 0.8)
train = data.iloc[:index].copy()
test = data.iloc[index:].copy()

to_remove = ['flow_id', 'source_ip', 'destination_ip', 'timestamp']
train.drop(to_remove, errors='ignore', inplace=True)
test.drop(to_remove, errors='ignore', inplace=True)

x_train = train.drop(columns=['malicious'])
y_train = train['malicious']

x_test = test.drop(columns=['malicious'])
y_test = test['malicious']